**1. Reading movie data with Pandas**

In [12]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest.zip

!unzip ml-latest.zip

import pandas as pd

movies = pd.read_csv("ml-latest/movies.csv")

movies


--2025-12-14 17:14:30--  https://files.grouplens.org/datasets/movielens/ml-latest.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 350896731 (335M) [application/zip]
Saving to: ‘ml-latest.zip.1’

ml-latest.zip.1     100%[===================>] 334.64M  57.3MB/s    in 5.5s    

2025-12-14 17:14:36 (61.2 MB/s) - ‘ml-latest.zip.1’ saved [350896731/350896731]

Archive:  ml-latest.zip
replace ml-latest/tags.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ml-latest/tags.csv      
replace ml-latest/links.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ml-latest/links.csv     
replace ml-latest/README.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ml-latest/README.txt    
replace ml-latest/ratings.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ml-latest/ratings.csv   y
y

replace ml-lat

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
86532,288967,State of Siege: Temple Attack (2021),Action|Drama
86533,288971,Ouija Japan (2021),Action|Horror
86534,288975,The Men Who Made the Movies: Howard Hawks (1973),Documentary
86535,288977,Skinford: Death Sentence (2023),Crime|Thriller


**2. Cleaning movie titles with regex**

In [13]:
import re

def clean_title(title):
    return re.sub("[^a-zA-Z0-9 ]", "", title)

movies["clean_title"] = movies["title"].apply(clean_title)

In [14]:
movies

,movieId,title,genres,clean_title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji 1995
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men 1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale 1995
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II 1995
...,...,...,...,...
86532,288967,State of Siege: Temple Attack (2021),Action|Drama,State of Siege Temple Attack 2021
86533,288971,Ouija Japan (2021),Action|Horror,Ouija Japan 2021
86534,288975,The Men Who Made the Movies: Howard Hawks (1973),Documentary,The Men Who Made the Movies Howard Hawks 1973
86535,288977,Skinford: Death Sentence (2023),Crime|Thriller,Skinford Death Sentence 2023


**3. Creating a TFIDF Matrix**

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(1,2))

tfidf = vectorizer.fit_transform(movies["clean_title"])

**4. Creating a search function**

In [16]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def search(title):
    title = clean_title(title)
    query_vec = vectorizer.transform([title])
    similarity = cosine_similarity(query_vec, tfidf).flatten()
    indices = np.argpartition(similarity, -5)[-5:]
    results = movies.iloc[indices].iloc[::-1]
    return results

In [17]:
x = search("Toy Story 1995")
x

,movieId,title,genres,clean_title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
60793,201588,Toy Story 4 (2019),Adventure|Animation|Children|Comedy,Toy Story 4 2019
14815,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,Toy Story 3 2010
3021,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 2 1999
20507,106022,Toy Story of Terror (2013),Animation|Children|Comedy,Toy Story of Terror 2013


**5. Building an interactive search box**

In [18]:
import ipywidgets as widgets
from IPython.display import display

movie_input = widgets.Text(
    value='Toy Story',
    description='Movie Title:',
    disabled=False
)
movie_list = widgets.Output()

def on_type(data):
    with movie_list:
        movie_list.clear_output()
        title = data["new"]
        if len(title) > 5:
            display(search(title))

movie_input.observe(on_type, names='value')

display(movie_input, movie_list)

Text(value='Toy Story', description='Movie Title:')

Output()

**6. Reading in movie ratings data**

In [19]:
ratings = pd.read_csv("ml-latest/ratings.csv")
ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,1225734739
1,1,110,4.0,1225865086
2,1,158,4.0,1225733503
3,1,260,4.5,1225735204
4,1,356,5.0,1225735119
...,...,...,...,...
33832157,330975,8340,2.0,1091583256
33832158,330975,8493,2.5,1091585709
33832159,330975,8622,4.0,1091581777
33832160,330975,8665,3.0,1091581765


In [20]:
ratings.dtypes

,0
userId,int64
movieId,int64
rating,float64
timestamp,int64


**7. Finding users who liked the same movie**

In [21]:
movie_id = 1

In [22]:
similar_users = ratings[(ratings["movieId"] == movie_id) & (ratings["rating"] > 4)]["userId"].unique()

similar_users

array([     2,     12,     24, ..., 330947, 330951, 330955])

**8. Finding other movies they like**

In [23]:
similar_user_recs = ratings[(ratings["userId"].isin(similar_users)) & (ratings["rating"] > 4)]["movieId"]

similar_user_recs

,movieId
62,1
67,17
69,21
72,34
73,36
...,...
33830278,786
33830279,788
33830280,802
33830281,805


**9. Finding the movies that have greater than 10%**

In [24]:
similar_user_recs = similar_user_recs.value_counts() / len(similar_users)

In [25]:
similar_user_recs = similar_user_recs[similar_user_recs > .1]
similar_user_recs

,count
movieId,
1,1.000000
318,0.424204
260,0.385136
356,0.357989
296,0.345989
...,...
1208,0.104182
1387,0.103435
3996,0.102294


**10. Finding how much all users like movies**

In [26]:
all_users = ratings[(ratings["movieId"].isin(similar_user_recs.index)) & (ratings["rating"] > 4)]

all_users

,userId,movieId,rating,timestamp
3,1,260,4.5,1225735204
4,1,356,5.0,1225735119
7,1,1036,5.0,1225735626
12,1,1210,4.5,1225735210
14,1,1291,5.0,1225734809
...,...,...,...,...
33831754,330974,4963,5.0,1457563122
33831755,330974,4993,4.5,1457563097
33831759,330974,5952,4.5,1457563120
33831765,330974,7153,4.5,1457563106


In [27]:
all_user_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

all_user_recs

,count
movieId,
318,0.314060
296,0.235714
2571,0.219088
356,0.203730
2959,0.192585
...,...
1073,0.037133
134853,0.036077
1387,0.035993


In [28]:
rec_percentages = pd.concat([similar_user_recs, all_user_recs], axis=1)
rec_percentages.columns = ["similar", "all"]
rec_percentages

,similar,all
movieId,,
1,1.000000,0.101681
318,0.424204,0.314060
260,0.385136,0.186964
356,0.357989,0.203730
296,0.345989,0.235714
...,...,...
1208,0.104182,0.064868
1387,0.103435,0.035993
3996,0.102294,0.048086


In [29]:
rec_percentages["score"] = rec_percentages["similar"] / rec_percentages["all"]
rec_percentages = rec_percentages.sort_values("score", ascending=False)
rec_percentages

,similar,all,score
movieId,,,
1,1.000000,0.101681,9.834678
3114,0.267262,0.042281,6.321030
78499,0.158516,0.030072,5.271241
4886,0.234843,0.060076,3.909116
6377,0.223905,0.059460,3.765670
...,...,...,...
858,0.259787,0.182183,1.425963
318,0.424204,0.314060,1.350710
2959,0.255852,0.192585,1.328519


In [30]:
rec_percentages.head(10).merge(movies, left_index=True, right_on="movieId")

,similar,all,score,movieId,title,genres,clean_title
0,1.000000,0.101681,9.834678,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
3021,0.267262,0.042281,6.321030,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 2 1999
14815,0.158516,0.030072,5.271241,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,Toy Story 3 2010
4781,0.234843,0.060076,3.909116,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,Monsters Inc 2001
6259,0.223905,0.059460,3.765670,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy,Finding Nemo 2003
580,0.197978,0.052843,3.746544,588,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,Aladdin 1992
359,0.242751,0.067141,3.615561,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,Lion King The 1994
8248,0.204706,0.057687,3.548532,8961,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,Incredibles The 2004
587,0.161231,0.046010,3.504261,595,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX,Beauty and the Beast 1991
1047,0.127710,0.037133,3.439276,1073,Willy Wonka & the Chocolate Factory (1971),Children|Comedy|Fantasy|Musical,Willy Wonka the Chocolate Factory 1971


**11. Building a recommendation function**

In [31]:
def find_similar_movies(movie_id):
    similar_users = ratings[(ratings["movieId"] == movie_id) & (ratings["rating"] > 4)]["userId"].unique()
    similar_user_recs = ratings[(ratings["userId"].isin(similar_users)) & (ratings["rating"] > 4)]["movieId"]

    similar_user_recs = similar_user_recs.value_counts() / len(similar_users)
    similar_user_recs = similar_user_recs[similar_user_recs > .10]

    all_users = ratings[(ratings["movieId"].isin(similar_user_recs.index)) & (ratings["rating"] > 4)]
    all_user_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

    rec_percentages = pd.concat([similar_user_recs, all_user_recs], axis=1)
    rec_percentages.columns = ["similar", "all"]

    rec_percentages["score"] = rec_percentages["similar"] / rec_percentages["all"]

    rec_percentages = rec_percentages.sort_values("score", ascending=False)
    return rec_percentages.head(10).merge(movies, left_index=True, right_on="movieId")[["score", "title", "genres"]]

**12. Creating an interactive recommendation widget**

In [32]:
import ipywidgets as widgets
from IPython.display import display

movie_name_input = widgets.Text(
    value='Toy Story',
    description='Movie Title:',
    disabled=False
)
recommendation_list = widgets.Output()

def on_type(data):
    with recommendation_list:
        recommendation_list.clear_output()
        title = data["new"]
        if len(title) > 5:
            results = search(title)
            movie_id = results.iloc[0]["movieId"]
            display(find_similar_movies(movie_id))

movie_name_input.observe(on_type, names='value')

display(movie_name_input, recommendation_list)

Text(value='Toy Story', description='Movie Title:')

Output()

In [25]:
from joblib import dump

dump(vectorizer, "vectorizer.joblib")
dump(tfidf, "tfidf.joblib")

movies.to_csv("movies_clean.csv", index=False)